# 第 10 讲：线性规划与资源分配

> 用 scipy.optimize.linprog 重构线性规划教学，输出最优生产方案和可行域图。

## 课件导学

**任务情境**：资源分配问题要先写清决策变量、目标和约束，再交给求解器。

**关键概念**

- 决策变量
- 目标函数
- 不等式约束
- 可行域
- 最优解

**本讲产物**

- lp_solution.csv
- lp_summary.csv
- lp_feasible_region.png

## 资料链接与阅读抓手

下面这些链接优先选官方文档或稳定资料。课前先看标题和示例，课堂中遇到参数或概念再回查。

- [SciPy linprog 文档](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linprog.html)：学习线性规划的矩阵输入格式。
- [HiGHS 求解器](https://highs.dev/)：了解 SciPy 默认高性能线性优化求解器来源。
- [Matplotlib subplots 文档](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html)：把可行域和最优点画清楚。

## 使用说明

- 先从上到下运行全部单元。
- 每讲都会生成一个 `outputs/lesson-xx/` 目录，用于保存图表、表格或报告片段。
- 示例数据均为教学用合成数据，真实比赛中应替换为题目附件数据。

In [ ]:
LESSON_ID = "lesson-10"

from pathlib import Path
import os
import math
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_ROOT = Path(os.environ.get("COURSE_OUTPUT_DIR", REPO_ROOT / "outputs")) / LESSON_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
print(f"Output directory: {OUTPUT_ROOT}")

## 可运行示例与讲解路线

In [ ]:
from scipy.optimize import linprog

# maximize 3*x1 + 5*x2 -> minimize negative profit
c = [-3, -5]
A = [[2, 1], [1, 3], [0, 1]]
b = [100, 150, 40]
bounds = [(0, None), (0, None)]
res = linprog(c, A_ub=A, b_ub=b, bounds=bounds, method="highs")
solution = pd.DataFrame({
    "product": ["A", "B"],
    "quantity": res.x,
})
summary = pd.DataFrame({"max_profit": [-res.fun], "success": [res.success], "message": [res.message]})
solution.to_csv(OUTPUT_ROOT / "lp_solution.csv", index=False, encoding="utf-8-sig")
summary.to_csv(OUTPUT_ROOT / "lp_summary.csv", index=False, encoding="utf-8-sig")
solution, summary

In [ ]:
x = np.linspace(0, 80, 300)
y1 = 100 - 2*x
y2 = (150 - x) / 3
y3 = np.full_like(x, 40)
y = np.minimum.reduce([y1, y2, y3])
y = np.maximum(y, 0)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x, y1, label="2x1 + x2 <= 100")
ax.plot(x, y2, label="x1 + 3x2 <= 150")
ax.plot(x, y3, label="x2 <= 40")
ax.fill_between(x, 0, y, where=y>=0, alpha=0.25)
ax.scatter(res.x[0], res.x[1], color="red", s=80, label="optimum")
ax.set_xlim(0, 80)
ax.set_ylim(0, 60)
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("Linear programming feasible region")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "lp_feasible_region.png", dpi=160)
plt.show()

## 实战环节

**课堂任务**

- 把利润系数改成另一组值。
- 新增一个资源约束。
- 解释最优点为什么落在约束边界上。

**课后挑战**：将二维问题扩展到三种产品，并用表格解释结果而不是画可行域。

**验收清单**

- 变量、目标、约束是否一一对应
- 是否检查 res.success
- 最优解解释是否有业务含义

In [ ]:
lesson_resources = [{'title': 'SciPy linprog 文档', 'url': 'https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linprog.html', 'reading_note': '学习线性规划的矩阵输入格式。'}, {'title': 'HiGHS 求解器', 'url': 'https://highs.dev/', 'reading_note': '了解 SciPy 默认高性能线性优化求解器来源。'}, {'title': 'Matplotlib subplots 文档', 'url': 'https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html', 'reading_note': '把可行域和最优点画清楚。'}]
lesson_deliverables = ['lp_solution.csv', 'lp_summary.csv', 'lp_feasible_region.png']
lesson_checklist = ['变量、目标、约束是否一一对应', '是否检查 res.success', '最优解解释是否有业务含义']

pd.DataFrame(lesson_resources).to_csv(OUTPUT_ROOT / "lesson_resources.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"deliverable": lesson_deliverables}).to_csv(OUTPUT_ROOT / "lesson_deliverables.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"check_item": lesson_checklist}).to_csv(OUTPUT_ROOT / "lesson_checklist.csv", index=False, encoding="utf-8-sig")
print("Saved courseware resource map, deliverables, and checklist.")